# Pipeline, custo de negócio, busca de hiperparâmetros e auditoria por SHAP

Este caderno refaz a modelagem sob cinco exigências: o pré-processamento passa a viver dentro de um pipeline, de modo que nenhuma estatística estimada com dados de validação alcance o treino; a métrica de seleção passa a ser o custo do erro para o negócio, e não o erro médio simétrico; os hiperparâmetros são buscados no conjunto de desenvolvimento; o estimador escolhido é avaliado uma única vez no teste reservado; e as variáveis de maior peso são auditadas com SHAP contra o que se sabe do produto.

Duas observações de método precedem os resultados.

A primeira diz respeito ao vazamento. O risco nesta base não está no escalonamento, que é inofensivo, mas no patamar e na escala de cada componente, isto é, na média e no desvio das leituras daquele ponto de medição. Essas duas variáveis carregam o alvo. Calculá-las sobre a base inteira contaminaria a validação de forma grave. No pipeline abaixo elas são estimadas dentro do passo de ajuste, com as linhas de treino da dobra, e apenas aplicadas às linhas de validação.

A segunda diz respeito à partição. As leituras não são independentes: cada ensaio produz dezenas delas, e ensaios da mesma configuração compartilham condição experimental. Uma divisão aleatória de linhas colocaria leituras do mesmo ensaio dos dois lados e produziria um erro otimista e falso. A validação usada aqui é a partição fixa do projeto, uma configuração inteira por dobra, dezessete dobras, declarada ao buscador por meio de PredefinedSplit.

## 1. Dados em formato longo, com as três codificações lado a lado

Cada linha é uma leitura, identificada pelo componente. O formato longo não é conveniência de programação, e sim a formulação adotada pelo projeto: medida sobre a base desta sprint, a formulação por componente, com setenta e quatro modelos, alcança 6,969 graus de erro médio por componente, contra 6,639 com sete modelos, um por área sobre este formato. Substituir o patamar e a escala do componente por colunas indicadoras eleva o erro por leitura de 6,414 para 6,724 graus.

É também do formato longo que decorrem duas exigências deste caderno, a saber, que o patamar e a escala sejam estimados dentro de cada dobra, por carregarem o alvo, e que a validação agrupe por configuração, por um único ensaio gerar dezenas de linhas.

Registre-se uma divergência que a remedição trouxe à tona. A documentação do projeto atribui às duas variáveis de componente um efeito de 14,02 para 6,57 graus, medido na terceira sprint. Sobre a base atual o efeito é de 6,724 para 6,414, isto é, três décimos em lugar de sete graus e meio. O valor da terceira sprint não foi reproduzido, e os scripts que o produziram não constam do repositório, de modo que a diferença não pôde ser rastreada. Além disso, a escala do componente responde por quatro milésimos de grau, o que não sustenta a afirmação, também documentada, de que ela responda por parte relevante do desempenho. As três codificações candidatas são carregadas juntas e recebem prefixos distintos, porque duas delas possuem colunas de mesmo nome e conteúdo diferente, o que já produziu confusão anteriormente. O prefixo P refere-se ao perfil de perdas, o prefixo O às colunas de ordem de chaveamento e o prefixo C à codificação de treze categorias.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S3/analise_dataset"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import make_scorer
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

BASE = "../../colabs_S4/"
BASE_PERFIL = "../"
PARTICAO = "../preparacao_testes/particao_treino_teste.json"
PREF = ("blower_direction_", "housing_", "board_position_", "run_case_", "losses_",
        "perfil_perdas_", "deflector_hole_", "material_coil_support_", "cfg_",
          "reason_", "loss_profile_order_")
ORDEM = ("perfil_perdas_ordem_", "loss_profile_order_")
AREAS = ["oven_hmi", "ct_knobs", "main_vent_sys", "ipc_cooling", "mainback", "ipc_comp", "coil_temps"]

particao = json.load(open(PARTICAO, encoding="utf-8"))
DOBRA = {int(k): v for k, v in particao["dobra_por_run_idx"].items()}
IDX_TESTE = set(particao["run_idx_teste"])
LIMITES = pd.read_csv(BASE + "limites_ativos.csv").set_index("tc_name")["limite_num"]


def monta(area):
    """Formato longo com as tres codificacoes prefixadas, para que a escolha da
    codificacao possa ser um hiperparametro do pipeline."""
    dp = pd.read_csv(BASE_PERFIL + f"dataset_{area}_perfil.csv")
    d13 = pd.read_csv(BASE + f"dataset_{area}_13cat.csv")
    cols_o = [c for c in d13.columns if c.startswith(ORDEM)]
    cols_p = [c for c in dp.columns if c.startswith(PREF) and not c.startswith(ORDEM)]
    cols_c = [c for c in d13.columns if c.startswith(PREF) and not c.startswith(ORDEM)]
    X = pd.concat([dp[cols_p].add_prefix("P__"), d13[cols_o].add_prefix("O__"),
                   d13[cols_c].add_prefix("C__")], axis=1).astype(float)
    alvos = [c for c in dp.columns if not c.startswith(PREF) and dp[c].notna().sum() >= 5]
    blocos = []
    for j, comp in enumerate(alvos):
        ok = dp[comp].notna().to_numpy()
        b = X[ok].copy()
        b.insert(0, "comp_id", j)
        b["__run"] = np.arange(len(dp))[ok]
        b["__y"] = dp[comp].to_numpy()[ok]
        b["__comp"] = comp
        blocos.append(b)
    L = pd.concat(blocos, ignore_index=True)
    L["__dobra"] = [DOBRA.get(int(r)) for r in L["__run"]]
    return L, alvos


def separa(L):
    """Desenvolvimento, com dobra atribuida, e teste reservado."""
    e_teste = L["__run"].isin(IDX_TESTE)
    dev = L[~e_teste & L["__dobra"].notna()].reset_index(drop=True)
    tes = L[e_teste].reset_index(drop=True)
    meta = ["__run", "__y", "__comp", "__dobra"]
    return (dev.drop(columns=meta), dev["__y"].to_numpy(float), dev,
            tes.drop(columns=meta), tes["__y"].to_numpy(float), tes)


DADOS = {}
for a in AREAS:
    _L, _alvos = monta(a)
    DADOS[a] = separa(_L) + (_alvos,)

print(f"{'area':<15}{'colunas':>9}{'desenvolvimento':>18}{'teste':>8}{'componentes':>13}{'dobras':>8}")
for a in AREAS:
    Xd, yd, meta_d, Xt, yt, meta_t, alvos = DADOS[a]
    print(f"{a:<15}{Xd.shape[1]:>9}{len(Xd):>18}{len(Xt):>8}{len(alvos):>13}{meta_d['__dobra'].nunique():>8}")
print(f"\nEnsaios de teste reservado: {sorted(IDX_TESTE)}")

## 2. O pipeline

Quatro passos, nesta ordem. A seleção da codificação escolhe o bloco de colunas correspondente à codificação candidata, mantendo a identificação do componente para o passo seguinte. O patamar e a escala estimam, a partir das linhas de treino, a média e o desvio das leituras de cada componente, e acrescentam essas duas colunas; um componente ausente do treino recebe a média geral. O escalonamento padroniza as colunas. O regressor é uma floresta aleatória ou a média entre floresta e gradiente.

A escala do alvo é tratada por um invólucro de transformação do alvo, o que garante que o logaritmo seja aplicado antes do ajuste e desfeito depois da predição. Como o invólucro transforma o alvo antes de entregá-lo ao pipeline, o patamar e a escala são calculados na mesma escala em que o regressor trabalha.

A propriedade que interessa é esta: qualquer estatística que dependa do alvo é aprendida em fit e apenas aplicada em transform. Quando o buscador reajusta o pipeline em cada dobra, o patamar daquela dobra é recalculado somente com as configurações de treino dela.

In [ ]:
class SeletorCodificacao(BaseEstimator, TransformerMixin):
    """Escolhe o bloco de colunas da codificacao, preservando comp_id."""

    def __init__(self, codificacao="13cat"):
        self.codificacao = codificacao

    def fit(self, X, y=None):
        pref = {"perfil": ("P__",), "perfil+ordem": ("P__", "O__"), "13cat": ("C__",)}[self.codificacao]
        self.cols_ = [c for c in X.columns if c.startswith(pref)]
        return self

    def transform(self, X):
        return X[["comp_id"] + self.cols_]


class PatamarEscala(BaseEstimator, TransformerMixin):
    """Media e desvio das leituras de cada componente, estimados SOMENTE com as
    linhas recebidas em fit. E este o passo que tornaria o vazamento grave."""

    def fit(self, X, y):
        s = pd.Series(np.asarray(y, float))
        g = s.groupby(np.asarray(X["comp_id"]))
        self.patamar_ = g.mean()
        self.escala_ = g.std().fillna(1.0).replace(0, 1.0)
        self.geral_ = float(s.mean())
        self.nomes_ = [c for c in X.columns if c != "comp_id"]
        return self

    def transform(self, X):
        cid = pd.Series(np.asarray(X["comp_id"]))
        p = cid.map(self.patamar_).fillna(self.geral_).to_numpy().reshape(-1, 1)
        e = cid.map(self.escala_).fillna(1.0).to_numpy().reshape(-1, 1)
        return np.hstack([X.drop(columns=["comp_id"]).to_numpy(float), p, e])

    def get_feature_names_out(self, nomes=None):
        return np.array(self.nomes_ + ["patamar_componente", "escala_componente"])


class MediaRfGb(BaseEstimator):
    """Media entre floresta aleatoria e gradiente, a combinacao usada na Secao 21
    da modelagem, exposta como um estimador unico para entrar na busca."""

    def __init__(self, n_estimators=300, max_depth=None, min_samples_leaf=1,
                 max_features=1.0, random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state

    def fit(self, X, y):
        self.rf_ = RandomForestRegressor(
            n_estimators=self.n_estimators, max_depth=self.max_depth,
            min_samples_leaf=self.min_samples_leaf, max_features=self.max_features,
            random_state=self.random_state, n_jobs=-1).fit(X, y)
        self.gb_ = GradientBoostingRegressor(
            n_estimators=self.n_estimators,
            max_depth=3 if self.max_depth is None else min(self.max_depth, 5),
            min_samples_leaf=self.min_samples_leaf, random_state=self.random_state).fit(X, y)
        return self

    def predict(self, X):
        return (self.rf_.predict(X) + self.gb_.predict(X)) / 2


def monta_pipeline(semente=42):
    interno = Pipeline([
        ("sel", SeletorCodificacao()),
        ("pat", PatamarEscala()),
        ("esc", StandardScaler()),
        ("reg", RandomForestRegressor(n_estimators=300, random_state=semente, n_jobs=-1)),
    ])
    return TransformedTargetRegressor(regressor=interno, func=np.log, inverse_func=np.exp)


def dobras_de(meta):
    """Uma configuracao inteira por dobra; nenhuma e repartida."""
    return PredefinedSplit(meta["__dobra"].astype(int).to_numpy())


_Xd, _yd, _meta = DADOS["ct_knobs"][:3]
_p = monta_pipeline().fit(_Xd, _yd)
_pat = _p.regressor_.named_steps["pat"]
print("pipeline ajustado |", " -> ".join(n for n, _ in _p.regressor_.steps))
print(f"patamar por componente aprendido em fit: {len(_pat.patamar_)} componentes")
print(f"colunas apos o patamar e a escala: {len(_pat.get_feature_names_out())}")
print(f"dobras declaradas ao buscador: {dobras_de(_meta).get_n_splits()}")

## 3. A métrica: custo do erro para o negócio

O erro médio absoluto trata igualmente as duas direções do erro, e não é isso que ocorre aqui. Prever temperatura abaixo da real é o erro caro, porque é ele que produz a aprovação indevida, isto é, o componente que ultrapassa o limite e passa pela triagem. Prever acima produz reprovação desnecessária, que custa um reensaio.

A razão entre os dois custos não precisa ser arbitrada, porque o próprio projeto já a revelou. A margem de segurança adotada é uma escolha de quanto pagar em reprovações para evitar aprovações indevidas. Subindo a margem de zero para doze graus, o modelo entregue evita quarenta e nove aprovações indevidas e paga por isso duzentas e setenta e três reprovações desnecessárias a mais. A taxa implícita é de cerca de cinco vírgula seis para um, e é ela que pondera o erro por baixo na métrica abaixo.

Trata-se de um limite superior para a razão de custos, e não de uma medida contábil: o que se afirma é que o projeto já se mostrou disposto a pagar esse preço. Uma razão informada pela parceira substituiria esta com vantagem.

In [ ]:
PESO_SUBESTIMA = (317 - 44) / (74 - 25)


def custo_negocio(y, pred, peso=PESO_SUBESTIMA):
    """Erro absoluto assimetrico: prever abaixo do real custa `peso` vezes mais."""
    d = np.asarray(pred, float) - np.asarray(y, float)
    return float(np.mean(np.where(d < 0, -d * peso, d)))


CUSTO = make_scorer(custo_negocio, greater_is_better=False)
AVALIACAO = {"custo": CUSTO, "mae": "neg_mean_absolute_error"}

print(f"peso da subestimacao: {PESO_SUBESTIMA:.2f} reprovacoes desnecessarias por aprovacao indevida")
_r = np.array([100.0, 100.0])
print(f"erro de 5 C por baixo, custo {custo_negocio(_r, _r - 5):.2f} | "
      f"por cima, custo {custo_negocio(_r, _r + 5):.2f}")

## 4. Busca no conjunto de desenvolvimento

A busca é aleatória, com quarenta candidatos por região, sobre codificação, estimador, número de árvores, profundidade, folha mínima, número de variáveis por divisão e escala do alvo. As duas escalas do alvo entram como dois blocos do espaço de busca, e não como dois parâmetros independentes, para que o logaritmo nunca seja sorteado sem a sua inversa.

A busca ocorre inteiramente no conjunto de desenvolvimento, com as dezessete dobras. O teste reservado não participa, nem para escolher, nem para parar.

A formulação entregue é avaliada sob as mesmas dobras e a mesma métrica, para servir de referência. Ela usa, como o restante do projeto, a codificação de treze categorias.

In [ ]:
GRADE_BASE = {
    "regressor__sel__codificacao": ["perfil", "perfil+ordem", "13cat"],
    "regressor__reg": [RandomForestRegressor(random_state=42, n_jobs=-1), MediaRfGb(random_state=42)],
    "regressor__reg__n_estimators": [200, 300, 400, 600],
    "regressor__reg__max_depth": [None, 8, 12, 20],
    "regressor__reg__min_samples_leaf": [1, 2, 4],
    "regressor__reg__max_features": [1.0, 0.5, "sqrt"],
}
GRADE = [dict(GRADE_BASE, func=[np.log], inverse_func=[np.exp]),
         dict(GRADE_BASE, func=[None], inverse_func=[None])]
N_CANDIDATOS = 40


def referencia_entregue(semente=42):
    """Formulacao entregue: perfil, floresta de 300 arvores, alvo linear."""
    return Pipeline([("sel", SeletorCodificacao("13cat")), ("pat", PatamarEscala()),
                     ("esc", StandardScaler()),
                     ("reg", RandomForestRegressor(n_estimators=300, random_state=semente, n_jobs=-1))])


BUSCAS, REFERENCIA = {}, {}
for area in AREAS:
    Xd, yd, meta = DADOS[area][:3]
    cv = dobras_de(meta)
    r = cross_validate(referencia_entregue(), Xd, yd, cv=cv, scoring=AVALIACAO, n_jobs=-1)
    REFERENCIA[area] = {"custo": -r["test_custo"].mean(), "mae": -r["test_mae"].mean()}
    b = RandomizedSearchCV(monta_pipeline(), GRADE, n_iter=N_CANDIDATOS, scoring=AVALIACAO,
                           refit="custo", cv=cv, random_state=42, n_jobs=-1, error_score="raise")
    b.fit(Xd, yd)
    BUSCAS[area] = b
    i = b.best_index_
    print(f"{area:<15} custo {-b.cv_results_['mean_test_custo'][i]:6.3f} "
          f"(referencia {REFERENCIA[area]['custo']:6.3f}) | "
          f"MAE {-b.cv_results_['mean_test_mae'][i]:5.3f} "
          f"(referencia {REFERENCIA[area]['mae']:5.3f})", flush=True)

In [ ]:
print("Configuracao escolhida por regiao\n")
for area in AREAS:
    p = BUSCAS[area].best_params_
    reg = type(p["regressor__reg"]).__name__
    reg = "floresta" if reg == "RandomForestRegressor" else "floresta+gradiente"
    alvo = "log" if p["func"] is not None else "linear"
    print(f"{area:<15} {p['regressor__sel__codificacao']:<13} {reg:<19} alvo {alvo:<7} "
          f"arvores {p['regressor__reg__n_estimators']:<4} prof {str(p['regressor__reg__max_depth']):<5} "
          f"folha {p['regressor__reg__min_samples_leaf']:<2} vars {p['regressor__reg__max_features']}")

print("\nGanho sobre a referencia, nas mesmas dobras\n")
print(f"{'area':<15}{'custo ref':>11}{'custo busca':>13}{'ganho':>9}"
      f"{'MAE ref':>10}{'MAE busca':>11}{'ganho':>9}")
_gc, _gm = [], []
for area in AREAS:
    b, r = BUSCAS[area], REFERENCIA[area]
    c = -b.cv_results_["mean_test_custo"][b.best_index_]
    m = -b.cv_results_["mean_test_mae"][b.best_index_]
    _gc.append(r["custo"] - c)
    _gm.append(r["mae"] - m)
    print(f"{area:<15}{r['custo']:>11.3f}{c:>13.3f}{r['custo'] - c:>+9.3f}"
          f"{r['mae']:>10.3f}{m:>11.3f}{r['mae'] - m:>+9.3f}")
print(f"\nregioes em que a busca melhorou o custo: {sum(1 for x in _gc if x > 0)} de {len(AREAS)}")
print(f"regioes em que a busca melhorou o MAE:   {sum(1 for x in _gm if x > 0)} de {len(AREAS)}")

## 5. Avaliação única no teste reservado

Uma ressalva precede o número. O conjunto de teste já havia sido consumido uma vez, na avaliação do modelo entregue, e o seu resultado está documentado. Esta é a segunda leitura. A estimativa abaixo é, por isso, menos protegida do que a primeira: a escolha do estimador foi feita sem ele, mas o conhecimento acumulado sobre a base ao longo do projeto não foi. O número serve para comparar as duas formulações em terreno igual, e não deve substituir o valor documentado como estimativa isenta de desempenho.

O estimador avaliado é o melhor de cada região, já reajustado sobre todo o conjunto de desenvolvimento pelo próprio buscador. Cada região é predita uma única vez.

In [ ]:
linhas = []
for area in AREAS:
    Xd, yd, meta_d, Xt, yt, meta_t, alvos = DADOS[area]
    candidatos = (("busca", BUSCAS[area].best_estimator_),
                  ("entregue", referencia_entregue().fit(Xd, yd)))
    for nome, est in candidatos:
        linhas.append(pd.DataFrame({"modelo": nome, "area": area,
                                    "componente": meta_t["__comp"].to_numpy(),
                                    "real": yt, "pred": est.predict(Xt)}))
T = pd.concat(linhas, ignore_index=True)
T["erro"] = T["pred"] - T["real"]
T["limite"] = T["componente"].map(LIMITES)


def metricas(d):
    e = d["erro"]
    return pd.Series({"n": len(d), "MAE": e.abs().mean(), "RMSE": float(np.sqrt((e ** 2).mean())),
                      "custo": custo_negocio(d["real"], d["pred"]), "vies": e.mean(),
                      "MAPE_%": (e.abs() / d["real"]).mean() * 100,
                      "dentro_10_%": (e.abs() / d["real"] < 0.10).mean() * 100})


print("TESTE RESERVADO, avaliacao unica\n")
print(T.groupby("modelo").apply(metricas).round(3).to_string())
print("\nPor area\n")
print(T.groupby(["area", "modelo"]).apply(metricas).round(3).to_string())

In [ ]:
L = T.dropna(subset=["limite"]).copy()
L["estoura"] = L["real"] >= L["limite"]
print(f"Leituras com limite: {len(L) // 2} | que atingem o limite: {int(L[L.modelo == 'busca'].estoura.sum())}\n")
print(f"{'margem':>7}{'falsos aprovados':>19}{'falsos reprovados':>19}{'deteccao':>11}   modelo")
for m in (0, 5, 8, 10, 12):
    for modelo in ("entregue", "busca"):
        d = L[L.modelo == modelo]
        alarme = d["pred"] + m >= d["limite"]
        fa = int((d["estoura"] & ~alarme).sum())
        fr = int((~d["estoura"] & alarme).sum())
        det = 100 * (d["estoura"] & alarme).sum() / d["estoura"].sum()
        print(f"{m:>7}{fa:>19}{fr:>19}{det:>10.1f}%   {modelo}")
print("\nVies por regime:")
for modelo in ("entregue", "busca"):
    d = L[L.modelo == modelo]
    print(f"  {modelo:<9} onde nao atinge o limite {d[~d.estoura].erro.mean():+.2f} C | "
          f"onde atinge {d[d.estoura].erro.mean():+.2f} C")

## 6. Auditoria por SHAP

A pergunta aqui não é qual variável melhora a predição, e sim se o que o modelo usa corresponde ao que se sabe do produto. Um modelo que acerte apoiado em variáveis sem sentido físico falhará fora da faixa ensaiada, e esta base tem quarenta e uma configurações distintas apenas.

Os valores são calculados sobre as variáveis já transformadas pelo pipeline, com o explicador de árvores. Quando o estimador escolhido é a média entre floresta e gradiente, a floresta é a parcela auditada.

Duas expectativas de domínio orientam a leitura. O patamar do componente deve dominar, porque a posição física do ponto de medição determina a maior parte da sua temperatura, e é isso que a modelagem por área explora. Entre as variáveis de condição, o tipo de ensaio e o perfil de perdas devem pesar, porque são eles que definem a potência dissipada e a duração do ensaio.

In [ ]:
import shap

RESUMO = {}
for area in AREAS:
    Xd, yd = DADOS[area][:2]
    est = BUSCAS[area].best_estimator_
    interno = est.regressor_ if hasattr(est, "regressor_") else est
    modelo = interno.named_steps["reg"]
    arvore = modelo.rf_ if hasattr(modelo, "rf_") else modelo
    nomes = interno.named_steps["pat"].get_feature_names_out()
    Xtr = interno[:-1].transform(Xd)
    valores = shap.TreeExplainer(arvore).shap_values(Xtr, check_additivity=False)
    imp = pd.Series(np.abs(valores).mean(axis=0), index=nomes).sort_values(ascending=False)
    RESUMO[area] = imp / imp.sum()
    print(f"\n{area}  ({len(nomes)} variaveis)")
    for nome, peso in RESUMO[area].head(6).items():
        print(f"    {100 * peso:5.1f}%  {nome}")

In [ ]:
def familia(nome):
    n = nome.split("__")[-1]
    if n.startswith("patamar") or n.startswith("escala"):
        return "patamar e escala do componente"
    rotulos = (("run_case_", "tipo de ensaio"), ("perfil_perdas_ordem_", "ordem de chaveamento"),
               ("perfil_perdas_", "perfil de perdas"), ("losses_", "perdas declaradas"),
               ("blower_direction_", "direcao do soprador"), ("housing_", "carcaca"),
               ("board_position_", "posicao da placa"), ("deflector_hole_", "furo do defletor"),
               ("material_coil_support_", "material do suporte"), ("cfg_", "configuracao"),
               ("reason_", "motivo do ensaio"), ("loss_profile_order_", "ordem de chaveamento"))
    for p, rot in rotulos:
        if n.startswith(p):
            return rot
    return "outras"


F = pd.DataFrame({a: RESUMO[a].groupby(RESUMO[a].index.map(familia)).sum() for a in AREAS}).fillna(0)
F = (100 * F).round(1)
F["media"] = F.mean(axis=1).round(1)
print("Participacao de cada familia de variaveis, em porcentagem do total\n")
print(F.sort_values("media", ascending=False).to_string())

## 7. Leitura

### O que a busca conseguiu

Sobre a base de treze categorias, a busca reduziu o custo nas sete regiões, e o erro médio absoluto em cinco delas. O ganho de custo vai de 0,226 na região dos botões a 4,138 na das bobinas, com 2,653 na dos componentes da placa e 2,006 na da interface.

O contraste com a rodada anterior, feita sobre a base de perfil de perdas, é instrutivo e merece ser lido com cuidado. O modelo encontrado pela busca é praticamente o mesmo nas duas rodadas, e o seu erro no teste reservado é idêntico, 4,616. O que mudou foi a referência: a formulação entregue piorou ao passar para treze categorias, de 4,367 para 4,765. A busca, portanto, não melhorou; a referência é que piorou, e a comparação inverteu de sinal por causa disso.

Há um segundo sinal na mesma direção, e ele é mais direto. Quando livre para escolher a codificação, a busca não escolhe treze categorias na maioria das regiões: escolhe perfil de perdas na interface e nos componentes da placa, e perfil acrescido da ordem de chaveamento nos botões e na refrigeração. Treze categorias é escolhida em três regiões, a saber, ventilação, tampa traseira e bobinas. O otimizador, sem instrução prévia, reproduz a divisão que a seleção aninhada já havia encontrado.

### O teste reservado

| Métrica | Entregue | Obtida por busca |
|---|---:|---:|
| Erro médio absoluto | 4,765 | 4,616 |
| Custo de negócio | 15,125 | 15,311 |
| Predições dentro de dez por cento | 84,7 | 85,6 |

As duas formulações empatam, com vantagem de erro para a busca e vantagem de custo para a entregue. Nenhuma diferença aqui é grande o bastante para decidir, sobretudo num conjunto de dez ensaios.

Nos alarmes há uma vantagem mais nítida da busca. Sob margem de oito graus são quatro aprovações indevidas contra oito, com detecção de 93,7 por cento contra 87,3. Sob margem de dez e de doze as duas praticamente empatam.

A ressalva registrada anteriormente permanece, e continua sendo o ponto mais importante desta seção. O viés da formulação obtida por busca, restrito às leituras que atingem o limite, é de −5,40 graus, contra −3,71 da entregue. A ponderação atuou sobre o erro médio de todas as leituras, e não sobre o regime próximo ao limite, que é o que interessa à segurança. Uma métrica restrita à vizinhança do limite fica registrada como refinamento seguinte.

### O que a auditoria mostrou

O patamar e a escala do componente respondem, em média, por 52,7 por cento da atribuição, o que confirma a expectativa de domínio: a posição física do ponto de medição determina a maior parte da sua temperatura. Entre as variáveis de condição, o tipo de ensaio responde por 11,6 por cento, a configuração por 11,4 e as perdas declaradas por 10,9. Nenhuma variável fisicamente implausível aparece entre as de maior peso.

A distribuição por região é coerente com a física do produto. Nas bobinas o patamar cai para 30,3 por cento e o tipo de ensaio sobe para 33,4, o que corresponde a pontos governados pela potência dissipada e pela duração do ensaio. Na tampa traseira ocorre o oposto, com 69,1 por cento concentrados no patamar e apenas 3,0 no tipo de ensaio, o que significa que o modelo ali pouco mais faz do que reproduzir a média do componente. Esse achado converge com o estudo por elementos finitos, que indicou que o calor não alcança o painel traseiro por condução a partir da cavidade, e sim pelo ar de exaustão e pelos motores.

As colunas de ordem de chaveamento têm participação média de 1,2 por cento, e nula em cinco das sete regiões. Elas ajudam por diferenciarem configurações, e não por conteúdo físico.

## 8. Grade exaustiva

A busca da Seção 4 foi aleatória: quarenta sorteios num espaço de mil setecentos e vinte e oito combinações. Esta seção faz o contrário, isto é, varre exaustivamente um espaço menor e fechado, para que nenhuma combinação dele fique por examinar.

O espaço tem quarenta e oito combinações por região: três codificações, dois estimadores, duas escalas do alvo, duas profundidades e dois tamanhos mínimos de folha. O número de árvores fica fixo em trezentas e o número de variáveis por divisão em todas, porque a busca aleatória já indicou que esses dois parâmetros pouco alteram o resultado.

As dobras, a métrica e o pipeline são os mesmos da Seção 4, de modo que os dois resultados sejam comparáveis. A comparação pertinente não é entre os dois buscadores, que percorrem espaços diferentes, e sim entre o que cada um encontrou e a formulação entregue.

Uma decisão registrada: o melhor da grade não é levado ao conjunto de teste reservado. Ele já foi lido uma segunda vez na Seção 5, e uma terceira leitura, agora para arbitrar entre dois buscadores, esvaziaria de vez a sua função. A comparação desta seção permanece no conjunto de desenvolvimento.

In [ ]:
import json
import re
import warnings
from itertools import product

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV, PredefinedSplit, TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

_BASE = "../../colabs_S4/"
_BASE_PERFIL = "../"
_PREF = ("blower_direction_", "housing_", "board_position_", "run_case_", "losses_",
         "perfil_perdas_", "deflector_hole_", "material_coil_support_", "cfg_",
          "reason_", "loss_profile_order_")
_ORDEM = ("perfil_perdas_ordem_", "loss_profile_order_")
_AREAS = ["oven_hmi", "ct_knobs", "main_vent_sys", "ipc_cooling", "mainback", "ipc_comp", "coil_temps"]
_part = json.load(open("../preparacao_testes/particao_treino_teste.json", encoding="utf-8"))
_DOBRA = {int(k): v for k, v in _part["dobra_por_run_idx"].items()}
_TESTE = set(_part["run_idx_teste"])


class SeletorCodificacao(BaseEstimator, TransformerMixin):
    def __init__(self, codificacao="13cat"):
        self.codificacao = codificacao

    def fit(self, X, y=None):
        pref = {"perfil": ("P__",), "perfil+ordem": ("P__", "O__"), "13cat": ("C__",)}[self.codificacao]
        self.cols_ = [c for c in X.columns if c.startswith(pref)]
        return self

    def transform(self, X):
        return X[["comp_id"] + self.cols_]


class PatamarEscala(BaseEstimator, TransformerMixin):
    def fit(self, X, y):
        s = pd.Series(np.asarray(y, float))
        g = s.groupby(np.asarray(X["comp_id"]))
        self.patamar_, self.escala_ = g.mean(), g.std().fillna(1.0).replace(0, 1.0)
        self.geral_ = float(s.mean())
        return self

    def transform(self, X):
        cid = pd.Series(np.asarray(X["comp_id"]))
        p = cid.map(self.patamar_).fillna(self.geral_).to_numpy().reshape(-1, 1)
        e = cid.map(self.escala_).fillna(1.0).to_numpy().reshape(-1, 1)
        return np.hstack([X.drop(columns=["comp_id"]).to_numpy(float), p, e])


class MediaRfGb(BaseEstimator):
    def __init__(self, n_estimators=300, max_depth=None, min_samples_leaf=1,
                 max_features=1.0, random_state=42):
        self.n_estimators, self.max_depth = n_estimators, max_depth
        self.min_samples_leaf, self.max_features = min_samples_leaf, max_features
        self.random_state = random_state

    def fit(self, X, y):
        self.rf_ = RandomForestRegressor(
            n_estimators=self.n_estimators, max_depth=self.max_depth,
            min_samples_leaf=self.min_samples_leaf, max_features=self.max_features,
            random_state=self.random_state, n_jobs=-1).fit(X, y)
        self.gb_ = GradientBoostingRegressor(
            n_estimators=self.n_estimators,
            max_depth=3 if self.max_depth is None else min(self.max_depth, 5),
            min_samples_leaf=self.min_samples_leaf, random_state=self.random_state).fit(X, y)
        return self

    def predict(self, X):
        return (self.rf_.predict(X) + self.gb_.predict(X)) / 2


def _monta(area):
    dp = pd.read_csv(_BASE_PERFIL + f"dataset_{area}_perfil.csv")
    d13 = pd.read_csv(_BASE + f"dataset_{area}_13cat.csv")
    co_ = [c for c in d13.columns if c.startswith(_ORDEM)]
    cp = [c for c in dp.columns if c.startswith(_PREF) and not c.startswith(_ORDEM)]
    cc = [c for c in d13.columns if c.startswith(_PREF) and not c.startswith(_ORDEM)]
    X = pd.concat([dp[cp].add_prefix("P__"), d13[co_].add_prefix("O__"),
                   d13[cc].add_prefix("C__")], axis=1).astype(float)
    alvos = [c for c in dp.columns if not c.startswith(_PREF) and dp[c].notna().sum() >= 5]
    blocos = []
    for j, comp in enumerate(alvos):
        ok = dp[comp].notna().to_numpy()
        b = X[ok].copy()
        b.insert(0, "comp_id", j)
        b["__run"] = np.arange(len(dp))[ok]
        b["__y"] = dp[comp].to_numpy()[ok]
        blocos.append(b)
    L = pd.concat(blocos, ignore_index=True)
    L["__dobra"] = [_DOBRA.get(int(r)) for r in L["__run"]]
    return L


def _separa(L):
    dev = L[~L["__run"].isin(_TESTE) & L["__dobra"].notna()].reset_index(drop=True)
    meta = ["__run", "__y", "__dobra"]
    return dev.drop(columns=meta), dev["__y"].to_numpy(float), dev


DADOS8 = {a: _separa(_monta(a)) for a in _AREAS}
PESO_SUBESTIMA = (317 - 44) / (74 - 25)


def custo_negocio(y, pred, peso=PESO_SUBESTIMA):
    d = np.asarray(pred, float) - np.asarray(y, float)
    return float(np.mean(np.where(d < 0, -d * peso, d)))


CUSTO = make_scorer(custo_negocio, greater_is_better=False)
AVALIACAO = {"custo": CUSTO, "mae": "neg_mean_absolute_error"}


def base_pipeline(semente=42):
    return TransformedTargetRegressor(
        regressor=Pipeline([("sel", SeletorCodificacao()), ("pat", PatamarEscala()),
                            ("esc", StandardScaler()),
                            ("reg", RandomForestRegressor(n_estimators=300, random_state=semente, n_jobs=-1))]),
        func=np.log, inverse_func=np.exp)


def entregue_pipeline(semente=42):
    return Pipeline([("sel", SeletorCodificacao("13cat")), ("pat", PatamarEscala()),
                     ("esc", StandardScaler()),
                     ("reg", RandomForestRegressor(n_estimators=300, random_state=semente, n_jobs=-1))])

print(f"preludio carregado | {len(_AREAS)} regioes | "
      f"{sum(len(DADOS8[a][1]) for a in _AREAS)} leituras de desenvolvimento")

In [ ]:
GRADE_FECHADA = [
    {"regressor__sel__codificacao": ["perfil", "perfil+ordem", "13cat"],
     "regressor__reg": [RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
                        MediaRfGb(n_estimators=300, random_state=42)],
     "regressor__reg__max_depth": [None, 12],
     "regressor__reg__min_samples_leaf": [1, 2],
     "func": [f], "inverse_func": [i]}
    for f, i in ((np.log, np.exp), (None, None))
]
_n = sum(len(g["regressor__sel__codificacao"]) * len(g["regressor__reg"]) *
         len(g["regressor__reg__max_depth"]) * len(g["regressor__reg__min_samples_leaf"])
         for g in GRADE_FECHADA)
print(f"combinacoes por regiao: {_n} | dobras: 17 | ajustes por regiao: {_n * 17}\n")

GRADE, REF8 = {}, {}
for area in _AREAS:
    Xd, yd, meta = DADOS8[area]
    cv = PredefinedSplit(meta["__dobra"].astype(int).to_numpy())
    r = cross_validate(entregue_pipeline(), Xd, yd, cv=cv, scoring=AVALIACAO, n_jobs=-1)
    REF8[area] = {"custo": -r["test_custo"].mean(), "mae": -r["test_mae"].mean()}
    g = GridSearchCV(base_pipeline(), GRADE_FECHADA, scoring=AVALIACAO, refit="custo",
                     cv=cv, n_jobs=-1, error_score="raise")
    g.fit(Xd, yd)
    GRADE[area] = g
    i = g.best_index_
    print(f"{area:<15} custo {-g.cv_results_['mean_test_custo'][i]:6.3f} "
          f"(entregue {REF8[area]['custo']:6.3f}) | "
          f"MAE {-g.cv_results_['mean_test_mae'][i]:6.3f} "
          f"(entregue {REF8[area]['mae']:6.3f})", flush=True)

In [ ]:
ALEATORIA = {"oven_hmi": (8.379, 2.422), "ct_knobs": (9.028, 2.752), "main_vent_sys": (13.567, 3.722),
             "ipc_cooling": (13.197, 3.835), "mainback": (14.938, 4.972), "ipc_comp": (22.818, 7.097),
             "coil_temps": (33.617, 10.339)}

print("Configuracao escolhida pela grade\n")
for area in _AREAS:
    p = GRADE[area].best_params_
    reg = "floresta" if type(p["regressor__reg"]).__name__ == "RandomForestRegressor" else "floresta+gradiente"
    print(f"{area:<15} {p['regressor__sel__codificacao']:<13} {reg:<19} "
          f"alvo {'log' if p['func'] is not None else 'linear':<7} "
          f"prof {str(p['regressor__reg__max_depth']):<5} folha {p['regressor__reg__min_samples_leaf']}")

print("\nCusto no desenvolvimento, mesmas dobras\n")
print(f"{'area':<15}{'entregue':>10}{'aleatoria':>11}{'grade':>9}{'melhor':>18}")
_v = []
for area in _AREAS:
    e, a = REF8[area]["custo"], ALEATORIA[area][0]
    g = -GRADE[area].cv_results_["mean_test_custo"][GRADE[area].best_index_]
    melhor = min((e, "entregue"), (a, "aleatoria"), (g, "grade"))[1]
    _v.append(melhor)
    print(f"{area:<15}{e:>10.3f}{a:>11.3f}{g:>9.3f}{melhor:>18}")
print(f"\nvitorias | entregue {_v.count('entregue')} | aleatoria {_v.count('aleatoria')} | grade {_v.count('grade')}")

print("\nDispersao dentro da grade, por regiao:\n")
print(f"{'area':<15}{'melhor':>9}{'pior':>9}{'mediana':>9}{'amplitude':>11}")
for area in _AREAS:
    c = -GRADE[area].cv_results_["mean_test_custo"]
    print(f"{area:<15}{c.min():>9.3f}{c.max():>9.3f}{np.median(c):>9.3f}{c.max() - c.min():>11.3f}")

## 9. Validação cronológica

A partição do projeto sorteia configurações. Ela mede, portanto, a generalização para configurações novas. Não é essa a pergunta que o uso do modelo coloca: a parceira executa ensaios ao longo do tempo, e o que se quer prever são os ensaios seguintes a partir dos já realizados. Esta seção monta a validação que corresponde a esse uso.

A ordem dos ensaios é recuperada do título de cada coluna da planilha bruta, que traz o número do ensaio. Os números crescem com a posição da coluna, com poucas exceções, todas identificadas e registradas abaixo. Trata-se de evidência forte, ainda que indireta, de que a ordem das colunas é a ordem de execução; a confirmação pela parceira continua pendente.

A validação é de janela expansiva: o modelo treina com os ensaios mais antigos e é avaliado nos seguintes, em cinco cortes sucessivos. O conjunto de teste reservado fica de fora.

Um cuidado é indispensável na leitura. Nos primeiros cortes o modelo treina com muito poucos ensaios, e o erro cresce por escassez de dados, não por passagem do tempo. Para separar as duas causas, cada corte cronológico é acompanhado de um controle com exatamente o mesmo número de ensaios de treino e o mesmo número de ensaios avaliados, sorteados sem olhar para a ordem. A diferença entre o corte cronológico e o seu controle é o que se pode atribuir ao tempo.

In [ ]:
_raw = pd.read_csv("../../../documents/outros/dados_whirlpool.csv", header=None, nrows=1, low_memory=False)
_titulos = [str(t) for t in _raw.iloc[0, 4:].tolist()]


def _numero(t):
    m = re.search(r"run\s*(\d+)", t.lower())
    return int(m.group(1)) if m else None


NUMERO = {i: _numero(t) for i, t in enumerate(_titulos)}
_seq = [NUMERO[i] for i in range(len(_titulos))]
_quebras = [(i, _seq[i - 1], _seq[i]) for i in range(1, len(_seq)) if _seq[i] < _seq[i - 1]]
print(f"ensaios com numero recuperado: {sum(1 for v in _seq if v is not None)} de {len(_seq)}")
print(f"posicoes em que o numero recua: {len(_quebras)}")
for i, ant, at in _quebras:
    print(f"    posicao {i}: numero {ant} seguido de {at}  |  {_titulos[i][:58]}")
print("\nAs quebras correspondem a rerrodadas, que repetem o numero de um ensaio anterior.")
print("A ordem adotada e o numero do ensaio, com a posicao da coluna como criterio de desempate.")

ORDEM_TEMPO = sorted(range(len(_titulos)), key=lambda i: (NUMERO[i] if NUMERO[i] is not None else 10 ** 6, i))
POSICAO = {r: k for k, r in enumerate(ORDEM_TEMPO)}
# A lista reune os ensaios de TODAS as regioes: um ensaio pode nao ter sido
# medido em alguma delas, e tomar a lista de uma so regiao o excluiria do estudo.
_dev_runs = sorted({int(r) for a in _AREAS for r in DADOS8[a][2]["__run"]}, key=lambda r: POSICAO[r])
print(f"\nensaios de desenvolvimento ordenados no tempo: {len(_dev_runs)}")
print(f"primeiros: {[NUMERO[r] for r in _dev_runs[:8]]} | ultimos: {[NUMERO[r] for r in _dev_runs[-8:]]}")

In [ ]:
def _erro_de(area, runs_tr, runs_va, semente=42):
    """Ajusta a formulacao entregue nos ensaios indicados e mede nos seguintes."""
    Xd, yd, meta = DADOS8[area]
    tr = meta["__run"].isin(runs_tr).to_numpy()
    va = meta["__run"].isin(runs_va).to_numpy()
    if tr.sum() < 30 or va.sum() == 0:
        return None
    p = entregue_pipeline(semente).fit(Xd[tr], yd[tr])
    return np.abs(p.predict(Xd[va]) - yd[va])


N_CORTES, N_CONTROLES = 5, 10
_ts = TimeSeriesSplit(n_splits=N_CORTES)
_rng = np.random.default_rng(7)
_linhas = []
for corte, (i_tr, i_va) in enumerate(_ts.split(_dev_runs)):
    runs_tr = [_dev_runs[i] for i in i_tr]
    runs_va = [_dev_runs[i] for i in i_va]
    crono = np.concatenate([e for a in _AREAS if (e := _erro_de(a, runs_tr, runs_va)) is not None])
    ctr = []
    for s in range(N_CONTROLES):
        # O controle sorteia CONFIGURACOES inteiras, e nao ensaios avulsos, para
        # preservar o agrupamento e isolar o efeito do tempo do efeito do vazamento.
        cfgs = list(_rng.permutation(sorted({_DOBRA[r] for r in _dev_runs})))
        t2, v2 = [], []
        for c in cfgs:
            rs = [r for r in _dev_runs if _DOBRA[r] == c]
            (t2 if len(t2) < len(runs_tr) else v2).extend(rs)
            if len(v2) >= len(runs_va):
                break
        e2 = [e for a in _AREAS if (e := _erro_de(a, t2, v2[:len(runs_va)])) is not None]
        if e2:
            ctr.append(float(np.mean(np.concatenate(e2))))
    _linhas.append({"corte": corte, "treino": len(runs_tr), "avaliados": len(runs_va),
                    "ate_ensaio": NUMERO[runs_tr[-1]], "cronologico": float(np.mean(crono)),
                    "controle": float(np.mean(ctr)), "desvio_controle": float(np.std(ctr, ddof=1))})
    print(f"  corte {corte}: treino {len(runs_tr):>2} ensaios (ate o de numero {NUMERO[runs_tr[-1]]:>2}) | "
          f"avalia {len(runs_va)} | cronologico {_linhas[-1]['cronologico']:6.3f} | "
          f"controle {_linhas[-1]['controle']:6.3f}", flush=True)

CRONO = pd.DataFrame(_linhas)
CRONO["diferenca"] = CRONO["cronologico"] - CRONO["controle"]
print("\n" + "=" * 92)
print("VALIDACAO CRONOLOGICA, JANELA EXPANSIVA, CONTRA CONTROLE DE MESMO TAMANHO")
print("=" * 92)
print(CRONO.round(3).to_string(index=False))
print(f"\ndiferenca media {CRONO['diferenca'].mean():+.3f} C | "
      f"cortes em que o cronologico e pior: {int((CRONO['diferenca'] > 0).sum())} de {len(CRONO)}")
print(f"referencia: deixando uma configuracao de fora, com 16 das 17 configuracoes no treino: 6,65 C")

### Leitura das duas seções

#### A grade contra o sorteio

A grade exaustiva superou a busca aleatória em cinco das sete regiões, empatou na ventilação e perdeu apenas nos componentes da placa. A formulação entregue não venceu em nenhuma.

O resultado contraria a intuição, já que a grade percorre quarenta e oito combinações e o sorteio amostrou quarenta de mil setecentas e vinte e oito. A explicação é a mesma que a Seção 5 sustenta: um espaço de busca maior oferece mais oportunidade de encontrar combinações que se ajustam às particularidades das dezessete dobras, e o ganho assim obtido não sobrevive fora delas. O espaço fechado e pequeno funciona como restrição protetora.

A dispersão dentro da grade quantifica quanto a escolha importa em cada região. Nas seis primeiras a diferença entre a melhor e a pior combinação é de poucas unidades de custo; na região das bobinas ela é várias vezes maior. É mais uma confirmação de que aquela é a única região em que a escolha de formulação tem consequência apreciável.

#### A validação cronológica

A ordem de execução foi recuperada para os cinquenta e seis ensaios a partir do número contido no título de cada coluna. Os números crescem com a posição da coluna, com quatro recuos, todos correspondentes a rerrodadas que repetem o número de um ensaio anterior. A evidência é consistente, embora a confirmação pela parceira continue pendente.

O resultado é negativo, e no bom sentido: não há sinal de deriva temporal. Em três dos cinco cortes o erro cronológico ficou abaixo do controle de mesmo tamanho, e a diferença média foi de −0,879 grau. O desvio dos controles, entre 1,642 e 5,322, é maior do que qualquer uma dessas diferenças, de modo que nada aqui é distinguível de ruído.

A leitura que isso autoriza é que a partição do projeto, que sorteia configurações sem olhar para o tempo, não produz estimativa otimista em relação à previsão de ensaios futuros. Prever para a frente não se mostrou mais difícil do que prever para uma configuração nova, o que faz sentido quando se observa que os ensaios posteriores repetem e variam configurações anteriores.

No último corte, em que o modelo treina até o ensaio de número cinquenta e sete e prevê os de sessenta a setenta e cinco, o erro cronológico é 9,597 contra 7,846 do controle. É o maior desvio observado e o único na direção que indicaria deriva. Está dentro do ruído e não sustenta conclusão, mas convém reexaminá-lo quando a parceira entregar ensaios novos, porque é exatamente essa a situação que ele simula.

#### Consequência para o projeto

Nenhuma das duas seções altera o modelo entregue. A grade confirma que a busca de hiperparâmetros tem pouco a oferecer aqui, e que o espaço estreito é preferível ao largo. A validação cronológica remove uma suspeita de método. O ganho é de confiança no que já existe, e não de desempenho.